# Business Problem

The Marketing and Sales Director of **BFLUBS** wants to segment the customer base into **five new divisions** using **data science**. This segmentation will be based on the **geolocation of customers**, aiming for better organization and more effective commercial strategies.

In addition, he requests the development of an **automated methodology** that allows classifying new customers into the correct divisions, based on the existing ones. This process should use **artificial intelligence** to ensure a precise and efficient allocation of new customers.

## Your task:
1. **Create five customer divisions** based on geolocation.
2. **Define a method** based on data science to classify new customers into the created divisions.

Describe the approach adopted, the techniques used, and the AI models applied to solve this problem.

# Import Data

In [3]:
import pandas as pd
import folium
import sklearn
import matplotlib.pyplot as plt

In [ ]:
customers = pd.read_csv('data/tbl_customers.csv', encoding='utf-8')
locations = pd.read_csv('data/tbl_locations.csv', encoding='utf-8')
plants = pd.read_csv('data/tbl_plants.csv', encoding='utf-8')

customer_fields = ['id_customer', 'ds_customer', 'id_location', 'sales_manager']
customers = customers[customer_fields].merge(
    locations,
    on='id_location',
    how='inner'
)
customers.head()

In [ ]:
plants

# Customers

In [ ]:
# Create a map centered on the average coordinates
map_ = folium.Map(
    location=[customers["latitude"].mean(), customers["longitude"].mean()],
    zoom_start=5
)

# Add a point for each customer
for _, row in customers.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        popup=row["ds_customer"],
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.6
    ).add_to(map_)

# Add plant markers (red industry icon)
for _, row in plants.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=row["ds_plant"],
        icon=folium.Icon(color="red", icon="industry", prefix="fa")
    ).add_to(map_)

# Display the map
map_

# Customer Classification

In [ ]:
from sklearn.cluster import KMeans

# Define the number of divisions (clusters)
num_clusters = 5

# Select only the geographic coordinates
X = customers[["latitude", "longitude"]]

# Apply KMeans for segmentation
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
customers["division_cluster"] = kmeans.fit_predict(X)

# Create a map centered on the average coordinates
map_ = folium.Map(
    location=[customers["latitude"].mean(), customers["longitude"].mean()],
    zoom_start=5
)

# Color list for the clusters
colors = ["red", "blue", "green", "purple", "orange"]

# Add customer points with colors based on cluster
for _, row in customers.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        color=colors[row["division_cluster"]],
        fill=True,
        fill_color=colors[row["division_cluster"]],
        fill_opacity=0.6
    ).add_to(map_)

# Add plant markers (dark blue industry icon)
# for _, row in plants.iterrows():
#     folium.Marker(
#         location=[row["latitude"], row["longitude"]],
#         popup=row["ds_plant"],
#         icon=folium.Icon(color="darkblue", icon="industry", prefix="fa")
#     ).add_to(map_)

# Map of Classified Customers

In [ ]:
map_

# Identifying the Ideal Number of Clusters / Groups

In [ ]:
# Define the range of clusters to test
num_clusters_range = range(1, 11)  # Testing from 1 to 10 clusters
inertia = []

# Calculate inertia for each number of clusters
for k in num_clusters_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(customers[["latitude", "longitude"]])
    inertia.append(kmeans.inertia_)

# Plot the Elbow Method chart
plt.figure(figsize=(8, 5))
plt.plot(num_clusters_range, inertia, marker='o', linestyle='-')
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia (Sum of Distances)")
plt.title("Elbow Method to Define the Ideal Number of Clusters")
plt.xticks(num_clusters_range)
plt.grid(True)
plt.show()